# BuildSafe-AI
##Feature Engineering & Transformation
**Project identifier:** Gima  ·  
> **Main explanation:** Machine-learning algorithms require numerical inputs. Raw categorical, date and text information must therefore be transformed into useful features.

**Responsibilities covered:** date feature engineering (`ISSUE_DATE` → Year, Month, Day, Day of Week, Quarter) · historical violation features · categorical encoding · text processing with TF-IDF.

**Input:** `data/processed/buildsafe_stage2_clean.csv` (Notebook 2). **Output:** `data/processed/buildsafe_features.csv` → used by Notebook 4.

> **About the outputs in this notebook.** Cells copied unchanged from the team's original notebooks keep the outputs from the original run on the real data.
> Cells that were newly written or reorganised for the viva have **no saved output** — run the notebook top-to-bottom (Kernel → Restart & Run All) before the evaluation so every number on screen is one you can explain.

In [1]:
import os, re
import numpy as np
import pandas as pd

PROCESSED_DIR = "../data/processed"
IN_PATH  = f"{PROCESSED_DIR}/buildsafe_stage2_clean.csv"
OUT_PATH = f"{PROCESSED_DIR}/buildsafe_features.csv"

# Building keys were saved as text ("UNKNOWN" is a valid value) — read them back as text
text_cols = {c: "string" for c in ["BIN", "BLOCK", "LOT", "RESPONDENT_CITY", "RESPONDENT_ZIP",
                                   "INFRACTION_CODE1", "SECTION_LAW_DESCRIPTION1", "VIOLATION_DESCRIPTION"]}
df = pd.read_csv(IN_PATH, dtype=text_cols, parse_dates=["ISSUE_DATE_DT"])

print("Loaded:", df.shape)
print(df.dtypes)

Loaded: (102467, 13)
BIN                                 string
BORO                                 int64
BLOCK                               string
LOT                                 string
ISSUE_DATE                           int64
SEVERITY                               str
VIOLATION_TYPE                         str
RESPONDENT_CITY                     string
RESPONDENT_ZIP                      string
VIOLATION_DESCRIPTION               string
INFRACTION_CODE1                    string
SECTION_LAW_DESCRIPTION1            string
ISSUE_DATE_DT               datetime64[us]
dtype: object


## 1. Date feature engineering

**What:** turn `ISSUE_DATE_DT` into five numeric features: Year, Month, Day, Day-of-week, Quarter.
**Why:** a model cannot read a date; but *when* a violation was issued may matter (seasonal inspection campaigns, weekday patterns, changes over time). Breaking the date into parts exposes those patterns as ordinary numbers.
**How:** the pandas `.dt` accessor.

| Feature | Meaning | Values |
|---|---|---|
| `ISSUE_YEAR` | year issued | 2025, 2026 |
| `ISSUE_MONTH` | month | 1 – 12 |
| `ISSUE_DAY` | day of month | 1 – 31 |
| `ISSUE_DAY_OF_WEEK` | Monday = 0 … Sunday = 6 | 0 – 6 |
| `ISSUE_QUARTER` | quarter of the year | 1 – 4 |

Only `ISSUE_DATE` is used: it is the one date that exists **at prediction time**. `SERVED_DATE` and `HEARING_DATE` were removed in Notebook 2 as post-issue information.

In [2]:
# Create temporal features from ISSUE_DATE

df["ISSUE_YEAR"] = df["ISSUE_DATE_DT"].dt.year
df["ISSUE_MONTH"] = df["ISSUE_DATE_DT"].dt.month
df["ISSUE_DAY"] = df["ISSUE_DATE_DT"].dt.day
df["ISSUE_DAY_OF_WEEK"] = df["ISSUE_DATE_DT"].dt.dayofweek
df["ISSUE_QUARTER"] = df["ISSUE_DATE_DT"].dt.quarter

print(
    df[
        [
            "ISSUE_DATE_DT",
            "ISSUE_YEAR",
            "ISSUE_MONTH",
            "ISSUE_DAY",
            "ISSUE_DAY_OF_WEEK",
            "ISSUE_QUARTER"
        ]
    ].head()
)

  ISSUE_DATE_DT  ISSUE_YEAR  ISSUE_MONTH  ISSUE_DAY  ISSUE_DAY_OF_WEEK  \
0    2026-06-01        2026            6          1                  0   
1    2025-08-05        2025            8          5                  1   
2    2026-06-24        2026            6         24                  2   
3    2025-12-03        2025           12          3                  2   
4    2026-07-22        2026            7         22                  2   

   ISSUE_QUARTER  
0              2  
1              3  
2              2  
3              4  
4              3  


In [3]:
# Do the date parts relate to severity at all? (row-normalised %)
pd.crosstab(df["ISSUE_DAY_OF_WEEK"], df["SEVERITY"], normalize="index").mul(100).round(1)

SEVERITY,CLASS - 1,CLASS - 2,CLASS - 3
ISSUE_DAY_OF_WEEK,,,
0,39.2,56.6,4.2
1,35.8,60.2,4.1
2,40.4,55.3,4.2
3,41.0,54.9,4.1
4,41.7,53.9,4.4
5,43.1,54.5,2.3
6,49.5,49.6,0.9


**Caveats to mention.** The data covers only ~20 months, so `ISSUE_YEAR` has just two values and month / quarter effects cannot be fully separated from a general time trend. `ISSUE_QUARTER` is derived from `ISSUE_MONTH` (partly redundant — harmless for the models used).

## 2. Historical violation features (building history)

**What:** for each violation, count how many violations the **same building** (`BIN`) had *before* it, in total and by severity class:
`PREVIOUS_VIOLATION_COUNT`, `PREVIOUS_CLASS1_COUNT`, `PREVIOUS_CLASS2_COUNT`, `PREVIOUS_CLASS3_COUNT`.
**Why:** a building with a bad record is likely to receive serious violations again. 19,917 of 38,743 buildings appear more than once (Notebook 1), so history is available for many rows.
**How:** count violations per (building, issue date), take a cumulative sum ordered by date, and subtract that day's own count — so only **strictly earlier dates** are counted.

### Why "strictly earlier"? (leakage safety)
If we counted violations issued the **same day**, a row could "see" its siblings' severity labels, which are not known when it is issued. Counting only earlier dates means the feature uses information that really was available at prediction time.

Worked example — the busiest building:

In [4]:
# Find the BIN with the most violations
top_bin = df["BIN"].value_counts().idxmax()

print("Selected BIN:", top_bin)
print("Number of violations:", (df["BIN"] == top_bin).sum())

print(
    df.loc[
        df["BIN"] == top_bin,
        ["BIN", "ISSUE_DATE", "SEVERITY", "VIOLATION_TYPE"]
    ].sort_values("ISSUE_DATE")
)

Selected BIN: UNKNOWN
Number of violations: 95
          BIN  ISSUE_DATE   SEVERITY VIOLATION_TYPE
6804  UNKNOWN    20250211  CLASS - 2   Construction
5719  UNKNOWN    20250221  CLASS - 2   Construction
8500  UNKNOWN    20250221  CLASS - 1   Construction
6380  UNKNOWN    20250225  CLASS - 2   Construction
8359  UNKNOWN    20250225  CLASS - 2   Construction
...       ...         ...        ...            ...
9277  UNKNOWN    20260807  CLASS - 2      Elevators
5947  UNKNOWN    20260901  CLASS - 2   Construction
7488  UNKNOWN    20260901  CLASS - 2   Construction
7260  UNKNOWN    20260901  CLASS - 2   Construction
9251  UNKNOWN    20260901  CLASS - 1   Construction

[95 rows x 4 columns]


In [5]:
top_bin = df["BIN"].value_counts().idxmax()

bin_history = (
    df[df["BIN"] == top_bin]
    .groupby("ISSUE_DATE")
    .agg(
        violations_on_date=("SEVERITY", "size"),
        class_1=("SEVERITY", lambda x: (x == "CLASS - 1").sum()),
        class_2=("SEVERITY", lambda x: (x == "CLASS - 2").sum()),
        class_3=("SEVERITY", lambda x: (x == "CLASS - 3").sum())
    )
    .sort_index()
)

print("BIN:", top_bin)
print(bin_history)

BIN: UNKNOWN
            violations_on_date  class_1  class_2  class_3
ISSUE_DATE                                               
20250211                     1        0        1        0
20250221                     2        1        1        0
20250225                     5        0        5        0
20250305                     1        0        1        0
20250313                     2        0        2        0
20250320                     1        0        1        0
20250324                     1        1        0        0
20250326                     1        0        1        0
20250408                     5        0        5        0
20250421                     2        0        2        0
20250602                     2        0        2        0
20250627                     1        1        0        0
20250710                     2        2        0        0
20250717                     4        0        4        0
20250723                     5        1        4        0
2

Read the table like this: on `20251211` the building had 10 new violations; before that day it had accumulated 5 earlier ones (1 × Class 1, 4 × Class 2), so those become that day's history features. The same-day violations are **not** counted.

Now the implementation for the whole dataset:

In [6]:
# Building-history features — only violations from EARLIER issue dates are counted
h = df[["BIN", "ISSUE_DATE_DT", "SEVERITY"]].copy()
for k in (1, 2, 3):
    h[f"c{k}"] = (h["SEVERITY"] == f"CLASS - {k}").astype(int)

daily = (h.groupby(["BIN", "ISSUE_DATE_DT"], as_index=False)
           .agg(n=("SEVERITY", "size"), c1=("c1", "sum"), c2=("c2", "sum"), c3=("c3", "sum"))
           .sort_values(["BIN", "ISSUE_DATE_DT"]))

for col, new in [("n", "PREVIOUS_VIOLATION_COUNT"), ("c1", "PREVIOUS_CLASS1_COUNT"),
                 ("c2", "PREVIOUS_CLASS2_COUNT"), ("c3", "PREVIOUS_CLASS3_COUNT")]:
    daily[new] = daily.groupby("BIN")[col].cumsum() - daily[col]      # cumulative up to today, minus today

history_columns = ["PREVIOUS_VIOLATION_COUNT", "PREVIOUS_CLASS1_COUNT",
                   "PREVIOUS_CLASS2_COUNT", "PREVIOUS_CLASS3_COUNT"]

df = df.merge(daily[["BIN", "ISSUE_DATE_DT"] + history_columns], on=["BIN", "ISSUE_DATE_DT"], how="left")

# Rows without a BIN have no identifiable building -> no reliable history.
# (Without this line all 95 'UNKNOWN' rows would be pooled together as if they were ONE building.)
df.loc[df["BIN"] == "UNKNOWN", history_columns] = 0
df[history_columns] = df[history_columns].fillna(0).astype(int)

print(df[history_columns].describe().T[["count", "mean", "std", "min", "max"]])

                             count      mean       std  min   max
PREVIOUS_VIOLATION_COUNT  102467.0  1.669250  3.504407  0.0  60.0
PREVIOUS_CLASS1_COUNT     102467.0  0.749217  1.991983  0.0  32.0
PREVIOUS_CLASS2_COUNT     102467.0  0.901305  2.116281  0.0  58.0
PREVIOUS_CLASS3_COUNT     102467.0  0.018728  0.156233  0.0   6.0


### Prove the feature is correct
A feature that is subtly wrong can leak. So we recompute the count the slow, obvious way for a random sample of 300 rows and compare.

In [7]:
sample = df.sample(300, random_state=42)
mismatches = 0
for _, r in sample.iterrows():
    if r["BIN"] == "UNKNOWN":
        expected = 0
    else:
        earlier = df[(df["BIN"] == r["BIN"]) & (df["ISSUE_DATE_DT"] < r["ISSUE_DATE_DT"])]
        expected = len(earlier)
    if expected != r["PREVIOUS_VIOLATION_COUNT"]:
        mismatches += 1

print("Rows checked:", len(sample), "| mismatches:", mismatches)
assert (df["PREVIOUS_CLASS1_COUNT"] + df["PREVIOUS_CLASS2_COUNT"] + df["PREVIOUS_CLASS3_COUNT"]
        == df["PREVIOUS_VIOLATION_COUNT"]).all(), "class counts must add up to the total"
print("Class counts add up to the total for every row.")

Rows checked: 300 | mismatches: 0
Class counts add up to the total for every row.


**Limitations.** (1) The data starts in Jan 2025, so early records have *artificially* low history (the true earlier violations are outside the window). (2) When the model is used on a new violation, the labels of earlier violations are known, so using them is legitimate at deployment time.

## 3. Text processing — scrub leaked labels, then TF-IDF

`VIOLATION_DESCRIPTION` is free text written by inspectors. It can be very informative (*"WORK WITHOUT A PERMIT"*, *"FAILURE TO FILE RETAINING WALL REPORT"*) — but investigation showed that many descriptions **literally contain the answer**, e.g. *"CLASS 1: CEASE USE…"* while the target is `CLASS - 1`.

### 3.1 What the leakage investigation found (from the team's text-leakage notebook)

| Finding | Count |
|---|---|
| Descriptions containing the substring "class" | 7,378 (7.20 %) |
| … that state a **severity** class (`CLASS 1/2/3`, `CLASS-1`, `CLASS ONE`, `CLASS I` …) | **4,273** (4.17 %) |
| … that mean a **legal occupancy** class (`CLASS A`, `CLASS B`, `CLASS J`) | 2,282 (2.23 %) |
| … that mean a **room** (`CLASSROOM`) | 659 (0.64 %) |

Only the severity mentions are leakage. Deleting the word "class" everywhere would also destroy the legitimate `CLASS A APARTMENT` and `CLASSROOM` signals — so the scrub targets **only** severity patterns.

### 3.2 The scrub
**What:** replace severity mentions with a space and tidy whitespace. **Why:** stops the model reading the label from the text. **How:** a regular expression. It is applied row by row and learns nothing from the data, so it cannot cause train/test leakage.

In [8]:
pat_severity  = r'(?i)\bclass\s*[-:#]?\s*(?:[123]|one|two|three|i{1,3})\b|\bclass[123]\b'
pat_occupancy = r'(?i)\bclass\s*[-_"\'\(\[]?\s*[abj]\b'
pat_classroom = r'(?i)\bclass\s*rooms?\b|\bclassrooms?\b'

def clean_description(text):
    if pd.isna(text):
        return ""
    cleaned = re.sub(pat_severity, " ", str(text))
    return re.sub(r"\s+", " ", cleaned).strip()

df["VIOLATION_DESCRIPTION_CLEAN"] = df["VIOLATION_DESCRIPTION"].apply(clean_description)

raw_txt   = df["VIOLATION_DESCRIPTION"].fillna("")
clean_txt = df["VIOLATION_DESCRIPTION_CLEAN"]
print("Severity mentions BEFORE:", int(raw_txt.str.contains(pat_severity, regex=True).sum()))
print("Severity mentions AFTER: ", int(clean_txt.str.contains(pat_severity, regex=True).sum()), "(must be 0)")
print("Occupancy 'CLASS A/B/J' preserved:", int(clean_txt.str.contains(pat_occupancy, regex=True).sum()))
print("'CLASSROOM' preserved:            ", int(clean_txt.str.contains(pat_classroom, regex=True).sum()))

Severity mentions BEFORE: 4273
Severity mentions AFTER:  0 (must be 0)
Occupancy 'CLASS A/B/J' preserved: 2282
'CLASSROOM' preserved:             659


In [9]:
# Three before / after examples
ex = df[raw_txt.str.contains(pat_severity, regex=True)].head(3)
for _, r in ex.iterrows():
    print("TARGET  :", r["SEVERITY"])
    print("BEFORE  :", r["VIOLATION_DESCRIPTION"][:150])
    print("AFTER   :", r["VIOLATION_DESCRIPTION_CLEAN"][:150])
    print()

TARGET  : CLASS - 2
BEFORE  : CLASS 2:FAILURE TO MAINTAIN BUILDING IN CODE COMPLIANT MANNER. SERVICE ELEVATOR EQUIPMENT AS PER BC3001.2 & 27-987. UPON INSPECTION, ELEVATOR PHONE IN
AFTER   : :FAILURE TO MAINTAIN BUILDING IN CODE COMPLIANT MANNER. SERVICE ELEVATOR EQUIPMENT AS PER BC3001.2 & 27-987. UPON INSPECTION, ELEVATOR PHONE INOPERATI

TARGET  : CLASS - 1
BEFORE  : CEASE USE ITEM: INOPERATIVE ZONE LOCK RESTRICTOR. (2.14.5.7.1.). CLASS 2 ITEMS: REPLACE WORN GIBS WHERE NEEDED (2.11.11.5.7). SECURE GIBS WHERE NEEDED
AFTER   : CEASE USE ITEM: INOPERATIVE ZONE LOCK RESTRICTOR. (2.14.5.7.1.). ITEMS: REPLACE WORN GIBS WHERE NEEDED (2.11.11.5.7). SECURE GIBS WHERE NEEDED (2.11.1

TARGET  : CLASS - 1
BEFORE  : CLASS 1(CU):FAILURE TO MAINTAIN BUILDING IN CODE COMPLIANT MANNER. SERVICE ELEVATOR EQUIPMENT AS PER BC3001.2 & 27-987. UPON INSPECTION, DOOR ZONE RES
AFTER   : (CU):FAILURE TO MAINTAIN BUILDING IN CODE COMPLIANT MANNER. SERVICE ELEVATOR EQUIPMENT AS PER BC3001.2 & 27-987. UPON INSPE

> **Honest note for the viva.** In the team's throwaway benchmark, scrubbing changed logistic-regression accuracy only from 79.50 % to 79.59 % (macro-F1 72.81 % → 73.00 %). So the text leak is *real and now removed*, but it is a small effect — the much larger effect is discussed in Notebook 4.

### 3.3 TF-IDF — how text becomes numbers
**What:** TF-IDF (*term frequency × inverse document frequency*) gives each word or word-pair a weight per description. **Why:** a model cannot read words. A word gets a **high** weight when it appears often in *this* description but rarely in *other* descriptions (so it is distinctive) and a **low** weight when it is common everywhere ("the", "to"). **How:** `TfidfVectorizer`. A small toy example makes the mechanism visible:

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

docs = ["FAILURE TO FILE RETAINING WALL REPORT",
        "WORK WITHOUT A PERMIT",
        "FAILURE TO MAINTAIN BUILDING"]

toy = TfidfVectorizer()
M = toy.fit_transform(docs)
pd.DataFrame(M.toarray(), columns=toy.get_feature_names_out(), index=["doc 1", "doc 2", "doc 3"]).round(2)

,building,failure,file,maintain,permit,report,retaining,to,wall,without,work
doc 1,0.00,0.33,0.44,0.00,0.00,0.44,0.44,0.33,0.44,0.00,0.00
doc 2,0.00,0.00,0.00,0.00,0.58,0.00,0.00,0.00,0.00,0.58,0.58
doc 3,0.56,0.43,0.00,0.56,0.00,0.00,0.00,0.43,0.00,0.00,0.00


Notice `failure` and `to` appear in two documents → lower weight; `permit` and `wall` appear in one → higher weight.

**Settings chosen for the real data** (the vectorizer is *created* here and *fitted on training data only* in Notebook 4):

| Setting | Value | Reason |
|---|---|---|
| `max_features` | 5000 | keep the 5,000 most useful terms → controls size and noise |
| `ngram_range` | (1, 2) | single words **and** two-word phrases ("without permit") |
| `min_df` | 2 | ignore terms that occur in only one document (typos, one-offs) |
| `max_df` | 0.95 | ignore terms present in more than 95 % of documents (no information) |
| `sublinear_tf` | True | dampens very long, repetitive descriptions |

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

print("TF-IDF vectorizer created.")

TF-IDF vectorizer created.


## 4. Categorical encoding

**What:** convert text categories into numbers with **one-hot encoding**. **Why:** the models need numbers, but the categories are *unordered* — if we coded `Boilers = 1, Construction = 2, Elevators = 3` the model would wrongly believe Elevators > Construction. One-hot gives each category its own 0/1 column, with no false ordering. **How:** `OneHotEncoder(handle_unknown="ignore")`.

In [12]:
from sklearn.preprocessing import OneHotEncoder

toy_df = pd.DataFrame({"VIOLATION_TYPE": ["Construction", "Elevators", "Boilers", "Construction"]})
enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(toy_df)

print(pd.DataFrame(enc.transform(toy_df), columns=enc.get_feature_names_out()))
print("\nA category never seen when fitting ('Signs') -> all zeros, no crash:")
print(enc.transform(pd.DataFrame({"VIOLATION_TYPE": ["Signs"]})))

   VIOLATION_TYPE_Boilers  VIOLATION_TYPE_Construction  \
0                     0.0                          1.0   
1                     0.0                          0.0   
2                     1.0                          0.0   
3                     0.0                          1.0   

   VIOLATION_TYPE_Elevators  
0                       0.0  
1                       1.0  
2                       0.0  
3                       0.0  

A category never seen when fitting ('Signs') -> all zeros, no crash:
[[0. 0. 0.]]


`handle_unknown="ignore"` matters in production: a brand-new city, ZIP or infraction code will eventually appear, and the encoder must not crash.

### Which columns are encoded — and which are not

In [13]:
candidate_cols = ["BORO", "VIOLATION_TYPE", "RESPONDENT_CITY", "RESPONDENT_ZIP",
                  "INFRACTION_CODE1", "SECTION_LAW_DESCRIPTION1", "BIN", "BLOCK", "LOT"]
card = pd.DataFrame({"unique_values": [df[c].nunique() for c in candidate_cols]}, index=candidate_cols)
card["share_of_rows (%)"] = (card["unique_values"] / len(df) * 100).round(2)
card

,unique_values,share_of_rows (%)
BORO,5,0.00
VIOLATION_TYPE,12,0.01
RESPONDENT_CITY,1313,1.28
RESPONDENT_ZIP,1412,1.38
INFRACTION_CODE1,326,0.32
SECTION_LAW_DESCRIPTION1,226,0.22
BIN,38744,37.81
BLOCK,9888,9.65
LOT,644,0.63


| Column | Distinct values | Decision | Reason |
|---|---|---|---|
| `BORO` | 5 | **one-hot** | small, meaningful location |
| `VIOLATION_TYPE` | 12 | **one-hot** | small, meaningful category |
| `INFRACTION_CODE1` | 326 | **one-hot** | the legal code of the violation; no missing values |
| `SECTION_LAW_DESCRIPTION1` | ~225 | **one-hot** | legal section wording |
| `RESPONDENT_CITY` / `RESPONDENT_ZIP` | ~1,200 / ~1,300 | **one-hot** | coarse location; rare values simply produce zeros |
| `BIN` / `BLOCK` / `LOT` | ~33,000 / ~9,500 / ~600 (train) | **not encoded** | thousands of values, most seen once or twice → would create thousands of sparse columns and encourage memorising individual buildings. The useful part of `BIN` — the building's record — is already captured by the history features. |
| `ISSUE_DATE`, `ISSUE_DATE_DT` | — | **not used** | replaced by the five date features |

In total, the six encoded columns become **3,011 one-hot columns** in the training split (Notebook 4).

**Flag for discussion:** a rule of thumb says "avoid high-cardinality one-hot" — we keep ~1,200 ZIP/city values because they are sparse-friendly and the models used handle sparse data well. Alternatives are frequency encoding or grouping rare values into `OTHER` (`OneHotEncoder(min_frequency=...)`).

## 5. Assemble and save the feature dataset

The **raw** `VIOLATION_DESCRIPTION` (which still contains the labels) is dropped here, so that only the scrubbed text can ever reach the modelling stage. It remains available in `buildsafe_stage2_clean.csv`.

In [14]:
df = df.sort_values("ISSUE_DATE_DT", kind="mergesort").reset_index(drop=True)

features_df = df.drop(columns=["VIOLATION_DESCRIPTION"])
print("Feature dataset:", features_df.shape)
print(list(features_df.columns))

features_df.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH)

Feature dataset: (102467, 22)
['BIN', 'BORO', 'BLOCK', 'LOT', 'ISSUE_DATE', 'SEVERITY', 'VIOLATION_TYPE', 'RESPONDENT_CITY', 'RESPONDENT_ZIP', 'INFRACTION_CODE1', 'SECTION_LAW_DESCRIPTION1', 'ISSUE_DATE_DT', 'ISSUE_YEAR', 'ISSUE_MONTH', 'ISSUE_DAY', 'ISSUE_DAY_OF_WEEK', 'ISSUE_QUARTER', 'PREVIOUS_VIOLATION_COUNT', 'PREVIOUS_CLASS1_COUNT', 'PREVIOUS_CLASS2_COUNT', 'PREVIOUS_CLASS3_COUNT', 'VIOLATION_DESCRIPTION_CLEAN']
Saved: ../data/processed/buildsafe_features.csv


## 6. Viva preparation — Member 3

**Q1. Why convert `ISSUE_DATE` into Year, Month, Day, Day-of-week and Quarter?** A model cannot use a date directly. Splitting it exposes yearly trends, seasonality and weekday patterns as numbers. Only the issue date is used because it is the only date known at prediction time.

**Q2. What is the historical-violation feature and why is it leakage-safe?** It counts a building's earlier violations (total and per class). Only strictly earlier dates are counted, so a row never sees same-day labels or the future. We verified it against a brute-force recomputation.

**Q3. Why one-hot and not label encoding?** The categories have no order. Label encoding would invent an order (3 > 2) the model would try to exploit.

**Q4. What if a category appears only in the test data?** `handle_unknown="ignore"` gives it all zeros instead of crashing.

**Q5. Why not one-hot `BIN`, `BLOCK`, `LOT`?** Tens of thousands of mostly-unique values → huge, sparse, memorisation-prone features. The information we want from `BIN` is provided by the history features.

**Q6. What is TF-IDF?** A weighting that rewards words that are frequent in one description but rare across all descriptions. Unigrams and bigrams, top 5,000 terms.

**Q7. Why scrub the description before TF-IDF?** 4,273 descriptions literally state the severity class. Leaving it would let the model read the answer. We remove only severity patterns, keeping `CLASS A` (occupancy) and `CLASSROOM`.

**Q8. Was the scrub the main cause of the very high SVM accuracy?** The team's own benchmark says no — the change was small (+0.09 points). Notebook 4 explains the likelier cause.

**Hand-off →** Notebook 4 loads `buildsafe_features.csv`, splits by date, fits the encoder and TF-IDF on training data only, and audits leakage.